### Import Dependencies and Set Reproducibility Controls

In [65]:
import numpy as np
import pandas as pd
from pathlib import Path
import spacy

SEED = 42
np.random.seed(SEED)

# Load spacy English model for NER and text processing
nlp = spacy.load("en_core_web_sm")
print(f"Loaded spacy model: {nlp.meta['name']} v{nlp.meta['version']}")

Loaded spacy model: core_web_sm v3.8.0


### Configure Local Paths

In [66]:
project_root = Path("..").resolve()
raw_csv_path = project_root / "data" / "raw_batch1" / "emails.csv"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data path:", raw_csv_path)
print("Processed dir:", processed_dir)

Project root: C:\Users\willd\Dissertation
Raw data path: C:\Users\willd\Dissertation\data\raw_batch1\emails.csv
Processed dir: C:\Users\willd\Dissertation\data\processed


### Load Batch1 Dataset

In [67]:
df = pd.read_csv(raw_csv_path)

print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
df.head(3)

Total rows: 266
Total columns: 2


,subject,body
0,KV6003 Assignment 2 Deadline,"Dear Dr Smith,\r\n\r\nI hope this email finds ..."
1,assignment help,hi\r\n\r\njust wondering when the assignment i...
2,KV5002 Coursework Submission Query,"Dear Lecturer,\r\n\r\nI wanted to check — do w..."


### Validate Dataset Schema and Data Quality

In [68]:
required_cols = ["subject", "body"]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print({
    "rows": len(df),
    "missing_subject": int(df["subject"].isna().sum()),
    "missing_body": int(df["body"].isna().sum()),
})

{'rows': 266, 'missing_subject': 0, 'missing_body': 0}


### Define PII Masking and Text Cleaning Functions

In [69]:
import re

PII_PATTERNS = {
    "EMAIL": r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b",
    "PHONE": r"\b(?:\+44\s?7\d{3}|07\d{3})\s?\d{3}\s?\d{3}\b",
    "STUDENT_ID": r"\b(?:[wW])?\d{7,9}\b",
}

SIGN_OFF_PATTERNS = [
    r"(?im)^kind regards,?.*$",
    r"(?im)^regards,?.*$",
    r"(?im)^thanks,?.*$",
    r"(?im)^many thanks,?.*$",
    r"(?im)^cheers,?.*$",
    r"(?im)^thx,?.*$",
    r"(?im)^ty,?.*$",
    r"(?im)^thank you,?.*$",
    r"(?im)^best wishes,?.*$",
]

PERSON_NAME_RE = re.compile(
    r"[A-Z][a-z]+(?:[-'][A-Za-z]+)?(?:\s+[A-Z][a-z]+(?:[-'][A-Za-z]+)?){1,2}"
)

PERSON_TITLE_RE = re.compile(r"^(Mr|Mrs|Ms|Miss|Dr|Prof)\.?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)?$")

NON_NAME_TERMS = {
    "reference", "request", "interruption", "studies", "attendance", "query",
    "urgent", "extension", "needed", "assignment", "group", "project", "concerns",
    "coursework", "results", "grade", "feedback", "support", "meeting", "module",
    "tutor", "personal", "lecture", "blackboard", "responsibilities", "consideration",
}

def should_mask_person(entity_text: str) -> bool:
    entity_text = re.sub(r"\s+", " ", entity_text).strip()
    if not entity_text:
        return False

    # Ignore likely IDs/codes and all-uppercase generic phrases.
    if re.search(r"\d", entity_text):
        return False
    if entity_text.isupper():
        return False

    words = [w.strip(".,;:!?()[]") for w in entity_text.split() if w.strip(".,;:!?()[]")]
    lower_words = {w.lower() for w in words}
    if lower_words & NON_NAME_TERMS:
        return False

    # Accept explicit titled names or conventional multi-token name forms.
    return bool(PERSON_TITLE_RE.fullmatch(entity_text) or PERSON_NAME_RE.fullmatch(entity_text))

def mask_pii(text: str) -> str:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)

    # Apply span-based NER masking on the original text first so indices stay valid.
    masked = text
    doc = nlp(text)
    entities = [
        (ent.start_char, ent.end_char)
        for ent in doc.ents
        if ent.label_ == "PERSON" and should_mask_person(ent.text)
    ]
    for start, end in sorted(entities, reverse=True):
        masked = masked[:start] + "[PERSON]" + masked[end:]

    # Then apply regex masking on the already NER-masked text.
    for label, pattern in PII_PATTERNS.items():
        masked = re.sub(pattern, f"[{label}]", masked)

    for pattern in SIGN_OFF_PATTERNS:
        masked = re.sub(pattern, "[SIGN_OFF]", masked)

    return masked

# removes quotes/URLs/sign-offs and normalizes whitespace
def clean_email_text(text: str) -> str:
    """Clean email text with quoted replies, URLs, and sign-offs removed."""
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)

    cleaned = text
    cleaned = re.sub(r"(?is)\n?On .*? wrote:.*$", " ", cleaned)
    cleaned = re.sub(r"(?im)^>.*$", " ", cleaned)
    cleaned = re.sub(r"https?://\S+|www\.\S+", " [URL] ", cleaned)

    for p in SIGN_OFF_PATTERNS:
        cleaned = re.sub(p, " ", cleaned)

    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

### Apply PII Masking and Text Cleaning

In [70]:
df_email = df.copy()
df_email["subject_masked"] = df_email["subject"].apply(mask_pii)
df_email["body_masked"] = df_email["body"].apply(mask_pii)
df_email["subject_clean"] = df_email["subject_masked"].apply(clean_email_text)
df_email["body_clean"] = df_email["body_masked"].apply(clean_email_text)
df_email["text_clean"] = (df_email["subject_clean"] + " [SEP] " + df_email["body_clean"]).str.strip()

print(f"\nProcessed {len(df_email)} emails")


Processed 266 emails


### Save Preprocessing Results

In [71]:
email_path = processed_dir / "email_masked_cleaned.csv"
df_email.to_csv(email_path, index=False)
print(email_path)

C:\Users\willd\Dissertation\data\processed\email_masked_cleaned.csv


### Load Processed and Prelabelled Data

In [72]:
label_dir = project_root / "data" / "batch1"
processed_path = processed_dir / "email_masked_cleaned.csv"

base_df = pd.read_csv(processed_path).copy()
label_files = sorted(label_dir.glob("*Emails.csv"))

if not label_files:
    raise FileNotFoundError(f"No prelabelled files found in {label_dir}")

print(f"Label files: {[f.name for f in label_files]}")

Label files: ['admEmails.csv', 'assEmails.csv', 'dupEmails.csv', 'extEmails.csv', 'fdbEmails.csv', 'mixEmails.csv', 'resEmails.csv', 'timEmails.csv']


### Build Labelled Dataset for Merge

In [73]:
label_parts = []
for f in label_files:
    d = pd.read_csv(f).copy()
    d["source_file"] = f.name
    label_parts.append(d)

labels_df = pd.concat(label_parts, ignore_index=True)
print(labels_df.shape)
labels_df.head(2)

(265, 8)


,email_id,subject,body,category,module,priority,multi_intent,source_file
0,ADM_001,Module Registration Issue,"Dear Dr Smith,\r\n\r\nI'm trying to register f...",admin,KV6003,high,False,admEmails.csv
1,ADM_002,changing modules,hi\r\n\r\ni want to drop kv4006 and switch to ...,admin,KV4006,high,False,admEmails.csv


### Apply Masking and Cleaning to Labelled Rows

In [74]:
for col in ["subject", "body"]:
    labels_df[col] = labels_df[col].fillna("").astype(str)

labels_df["subject_masked"] = labels_df["subject"].apply(mask_pii)
labels_df["body_masked"] = labels_df["body"].apply(mask_pii)
labels_df["subject_clean"] = labels_df["subject_masked"].apply(clean_email_text)
labels_df["body_clean"] = labels_df["body_masked"].apply(clean_email_text)
labels_df["text_clean"] = (labels_df["subject_clean"] + " [SEP] " + labels_df["body_clean"]).str.strip()

### Convert Categories to Label Lists

In [75]:
def parse_label_list(value: str):
    value = "" if pd.isna(value) else str(value)
    labels = [x.strip().lower() for x in value.split("|") if x.strip()]
    return list(dict.fromkeys(labels))

labels_df["label_list"] = labels_df["category"].apply(parse_label_list)

label_map = labels_df.groupby("text_clean", as_index=False).agg({
    "label_list": lambda rows: sorted(set(label for row in rows for label in row))
})

### Join Labels Onto Dataset

In [76]:
base_df = base_df.merge(label_map, on="text_clean", how="left")
base_df["label_list"] = base_df["label_list"].apply(lambda x: x if isinstance(x, list) else [])
base_df["n_labels"] = base_df["label_list"].str.len()

matched = int((base_df["n_labels"] > 0).sum())
print(f"Matched rows: {matched}/{len(base_df)} ({matched/len(base_df):.1%})")

Matched rows: 265/266 (99.6%)


### Build Taxonomy and Multi-Label Columns

In [77]:
taxonomy = sorted({label for labels in base_df["label_list"] for label in labels})
label_to_id = {label: idx for idx, label in enumerate(taxonomy)}

for label in taxonomy:
    base_df[f"y_{label}"] = base_df["label_list"].apply(lambda labels, t=label: int(t in labels))

base_df["label_ids"] = base_df["label_list"].apply(lambda labels: [label_to_id[l] for l in labels])

print("Taxonomy:", taxonomy)

Taxonomy: ['admin', 'assessment', 'extenuating_circumstances', 'feedback', 'resources', 'timetabling']


### Save Labelled Dataset and Taxonomy

In [78]:
target_path = processed_dir / "email_processed_labeled_multihot.csv"
taxonomy_path = processed_dir / "label_taxonomy.csv"

base_df.to_csv(target_path, index=False)
pd.DataFrame({
    "label": taxonomy,
    "label_id": [label_to_id[l] for l in taxonomy]
}).to_csv(taxonomy_path, index=False)

print(target_path)
print(taxonomy_path)

C:\Users\willd\Dissertation\data\processed\email_processed_labeled_multihot.csv
C:\Users\willd\Dissertation\data\processed\label_taxonomy.csv


### Create Reproducible Train/Validation/Test Split

In [79]:
SPLIT_SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

if not np.isclose(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 1.0):
    raise ValueError("Split ratios must sum to 1.0")

rng = np.random.default_rng(SPLIT_SEED)
indices = np.arange(len(base_df))
rng.shuffle(indices)

n_total = len(base_df)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

df_train = base_df.iloc[train_idx].reset_index(drop=True)
df_val = base_df.iloc[val_idx].reset_index(drop=True)
df_test = base_df.iloc[test_idx].reset_index(drop=True)

print(f"Rows total: {n_total}")
print(f"Train: {len(df_train)} ({len(df_train)/n_total:.1%})")
print(f"Validation: {len(df_val)} ({len(df_val)/n_total:.1%})")
print(f"Test: {len(df_test)} ({len(df_test)/n_total:.1%})")

Rows total: 266
Train: 186 (69.9%)
Validation: 39 (14.7%)
Test: 41 (15.4%)


### Save Split Artifacts for Modelling

In [80]:
split_dir = processed_dir / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

train_path = split_dir / "train.csv"
val_path = split_dir / "val.csv"
test_path = split_dir / "test.csv"

# Keep one label list for later training code
target_cols = [c for c in base_df.columns if c.startswith("y_")]

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path, index=False)
df_test.to_csv(test_path, index=False)

split_meta = pd.DataFrame([
    {"split": "train", "rows": len(df_train), "ratio": len(df_train) / len(base_df), "seed": SPLIT_SEED},
    {"split": "val", "rows": len(df_val), "ratio": len(df_val) / len(base_df), "seed": SPLIT_SEED},
    {"split": "test", "rows": len(df_test), "ratio": len(df_test) / len(base_df), "seed": SPLIT_SEED},
])
split_meta_path = split_dir / "split_metadata.csv"
split_meta.to_csv(split_meta_path, index=False)

print("Saved split files:")
print(train_path)
print(val_path)
print(test_path)
print(split_meta_path)
print(f"Target columns ({len(target_cols)}): {target_cols}")

Saved split files:
C:\Users\willd\Dissertation\data\processed\splits\train.csv
C:\Users\willd\Dissertation\data\processed\splits\val.csv
C:\Users\willd\Dissertation\data\processed\splits\test.csv
C:\Users\willd\Dissertation\data\processed\splits\split_metadata.csv
Target columns (6): ['y_admin', 'y_assessment', 'y_extenuating_circumstances', 'y_feedback', 'y_resources', 'y_timetabling']
